Cosine similarity

In [1]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

cat = np.array([1, 1, 0])
kitten = np.array([0.9, 1.1, 0])
car = np.array([0, 0, 1])

print("cat vs kitten:", cosine_similarity(cat, kitten))
print("cat vs car:", cosine_similarity(cat, car))

cat vs kitten: 0.995037190209989
cat vs car: 0.0


Sklearn

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity



# -----------------------------
# Vectorize (convert text → vectors)
# -----------------------------
vectorizer = TfidfVectorizer()
doc_vectors = vectorizer.fit_transform(documents)

# -----------------------------
# Search function
# -----------------------------
def search(query, k=3):
    query_vec = vectorizer.transform([query])

    similarities = cosine_similarity(query_vec, doc_vectors)[0]

    # get top k indices
    top_indices = similarities.argsort()[-k:][::-1]

    print("\nQuery:", query)
    print("\nTop results:\n")

    for i in top_indices:
        print(f"{documents[i]}  (score: {similarities[i]:.3f})")

# -----------------------------
# Try queries
# -----------------------------
search("How do computers learn from data?")
search("sports played in India")
search("space and planets")


Query: How do computers learn from data?

Top results:

Machine learning enables computers to learn from data.  (score: 0.724)
Pandas is a Python library for data analysis.  (score: 0.174)
Movies are a popular form of entertainment.  (score: 0.000)

Query: sports played in India

Top results:

Cricket is widely played in India and Australia.  (score: 0.594)
The Taj Mahal is located in India.  (score: 0.397)
Basketball is played with a ball and hoop.  (score: 0.238)

Query: space and planets

Top results:

Space exploration includes missions to Mars.  (score: 0.346)
NoSQL databases are flexible and scalable.  (score: 0.180)
Databases store and retrieve structured information.  (score: 0.174)


Chromadb OPENAI 

In [12]:
# import chromadb
# from chromadb.utils import embedding_functions
# import os

# # Use OpenAI embeddings (or replace with local model)
# openai_ef = embedding_functions.OpenAIEmbeddingFunction(
#     api_key=os.getenv("OPENAI_API_KEY"),
#     model_name="text-embedding-3-small"
# )

# # Create DB
# client = chromadb.Client()
# collection = client.get_or_create_collection(
#     name="demo",
#     embedding_function=openai_ef
# )

# # Add documents
# collection.add(
#     documents=documents,
#     ids=[str(i) for i in range(len(documents))]
# )

# # Query
# query = "How do computers learn from data?"

# results = collection.query(
#     query_texts=[query],
#     n_results=3
# )

# print("Query:", query)
# print("\nTop matches:")
# for doc in results["documents"][0]:
#     print("-", doc)

### Mini RAG-pipeine

- Here we build a mini RAG-Pipeline and integrate it with Chatgroq llm model. 
- So while passing a user query to the LLM, instead of directly generating the output it retrives relevant information form the document corpus and combines this context with the user query to pass as an input to the LLM. 
- Thus producing better outputs and eliminating hallucinations and black box reasonings.

In [35]:
import os
from langchain_groq.chat_models import ChatGroq

Groq_Token = os.getenv("CHAT_GROQ_API_KEY")

llm = ChatGroq(
    groq_api_key=Groq_Token,
    model_name="llama-3.1-8b-instant"
)

In [36]:
def non_rag_answer(query):
    response = llm.invoke(query)
    return response.content

In [37]:
def retrieve(query, k=3):
    query_vec = vectorizer.transform([query])
    similarities = cosine_similarity(query_vec, doc_vectors)[0]
    top_indices = similarities.argsort()[-k:][::-1]

    results = [(documents[i], similarities[i]) for i in top_indices]
    return results

In [38]:
def build_context(results):
    context = "\n".join([doc for doc, _ in results])
    return context

In [ ]:
# def generate_answer(query, results):
#     best_doc = results[0][0]
#     return f"Answer: {best_doc}"

In [40]:
def build_prompt(query, context):
    return f"""
You are an assistant that answers using the provided context.

Rules:
- Use the context to answer the question
- Do NOT copy a single sentence as the full answer
- Combine information if needed
- Explain in a clear way

Context:
{context}

Question:
{query}

Answer:
"""

In [41]:
def rag_pipeline(query, k=3):
    results = retrieve(query, k)
    context = build_context(results)
    prompt = build_prompt(query, context)
    response = llm.invoke(prompt)

    return {
        "query": query,
        "results": results,
        "context": context,
        "answer": response.content
    }

In [42]:
def rag_answer(query):
    return rag_pipeline(query)["answer"]

In [46]:
query = "How do computers learn from data? answer in paragraph of 50 words."

print("\n--- NON-RAG (Groq only) ---")
print(non_rag_answer(query))

print("\n--- RAG (TF-IDF + Groq) ---")
print(rag_answer(query))


--- NON-RAG (Groq only) ---
Computers learn from data through a process called machine learning. They use algorithms to analyze and identify patterns in large datasets. This process involves training, where the algorithm is fed examples of input and output pairs, allowing it to learn from the data and make predictions or decisions.

--- RAG (TF-IDF + Groq) ---
Computers learn from data through a process called machine learning. This process involves feeding large amounts of data into algorithms, which identify patterns and make predictions or decisions. The algorithms adjust and improve their performance with each iteration, enabling the computer to learn and adapt over time.


In [61]:
def llm_judge(query, non_rag, rag):
    prompt = f"""
You are an expert evaluator of AI answers.

Given a question and two answers:
- Answer A: Non-RAG (no context)
- Answer B: RAG (with retrieved context)

Evaluate both answers on:
1. Correctness (0-10)
2. Relevance (0-10)
3. Clarity (0-10)
4. Completeness (0-10)

Then give:
- Total score for A and B (out of 40)
- Which is better and why (1-2 sentences)

---

Question:
{query}

Answer A (Non-RAG):
{non_rag}

Answer B (RAG):
{rag}

Return format:
A_score: 
B_score:
Winner:
Reason:
"""
    response = llm.invoke(prompt)
    return response.content

In [62]:
print(llm_judge(query, non_rag_answer(query), rag_answer(query)))

A_score: 
- Correctness: 8
- Relevance: 9
- Clarity: 8
- Completeness: 7

Total score for A: 32

B_score: 
- Correctness: 9
- Relevance: 9
- Clarity: 8
- Completeness: 8

Total score for B: 34

Winner: B
Reason: Answer B is a better choice because it more directly addresses the question by explicitly mentioning the training process, which is a key aspect of how computers learn from data.
